In [1]:
import pandas as pd
import sys
sys.path.insert(0, "/Users/angelica/CNR/Git/pynarrative/pynarrative/src")
import pynarrative as pn
import altair as alt


In [2]:
df = pd.read_csv('source/Aquaculture_Exports.csv')
df = df[df['GEOGRAPHY_DESC'] == 'World']
df = df[df['UNIT_DESC'] == 'U.S.$']
#df = df[df['UNIT_DESC'] == 'KG']
df['DATE'] = pd.to_datetime(df['YEAR_ID'].astype(str) + '-' + df['TIMEPERIOD_ID'].astype(str) + '-1')
# print the max value of the 'DATE' column
color='#F27B6E'
#color='#0ABDFD'
domain = ['Salmon', 'Shrimp', 'Trout', 'Scallop', 'Other']
# from the dataframe filter out the following rows:
#   - the 'COMMODITY_DESC' value is not uppercase

df = df[~df['COMMODITY_DESC'].str.isupper()]

# add a new column to the dataframe called CATEGORY which is built as follows:
# - if the 'COMMODITY_DESC' value contains the word salmon or Salmon then the value is 'Salmon'
# - if the 'COMMODITY_DESC' value contains the word shrimp or Shrimp then the value is 'Shrimp'
# - if the 'COMMODITY_DESC' value contains the word trout or Trout then the value is 'Trout'
# - if the 'COMMODITY_DESC' value contains the word scallop or Scallop then the value is 'Scallop'
# - otherwise the value is 'Other'

df['CATEGORY'] = ['Salmon' if 'salmon' in x or 'Salmon' in x else 'Other' for x in df['COMMODITY_DESC']]
range = [color if 'Salmon' in x else 'lightgrey' for x in domain]

# group the dataframe by 'YEAR_ID' and 'COMMODITY_DESC' and sum the 'AMOUNT' column
# reset the index of the dataframe

df = df.groupby(['YEAR_ID', 'CATEGORY'])['AMOUNT'].sum().reset_index()

# remove year 2016 from the dataframe
df = df[df['YEAR_ID'] != 2016]


In [5]:
(pn.Story(df, width=600, height=400)
 .mark_line()
 .encode(
     x=alt.X('YEAR_ID:Q', title='Year', axis=alt.Axis(labelAngle=45, format='d')),
     y=alt.Y('AMOUNT:Q', title='Export Sales ($)'),
     color=alt.Color('CATEGORY:N', scale=alt.Scale(range=['#A7B3C7', '#1D4ED8']))
 )
 .add_context(
     text='Aquaculture includes farming aquatic animals and plants. This chart compares salmon export sales against all other aquaculture categories from 2000 to 2015.',
     position='left'
 )
 .add_title('USA Salmon Exports', 'FDA-regulated aquaculture exports (2000-2015)')
 .add_annotation(x=2002, y=470000000, text='Early-year weakness before sustained growth')
 .add_annotation(x=2008, y=820000000, text='Peak period for salmon export momentum')
 .add_next_steps(
     steps=['Strengthen cold-chain reliability', 'Expand premium market channels', 'Improve farm-level forecasting'],
     title='What can we do to avoid a new flop?'
 )
 .render()
 .configure_axis(grid=False)
 .configure_view(strokeWidth=0)
)


ValueError: add_annotation currently supports only quantitative x/y axes.